# MR-LSTM / MR-GRU

Market-rule-informed **recurrent** networks. Same market rules as MRINN, same
data pipeline, same metrics -- only the per-feature encoder changes, from a
Dense stack over a flat lag vector to an LSTM/GRU over a real time axis.

Splits, quantiles and batch size are kept identical to `MRINN_Tutorial.ipynb`
so results are directly comparable against the published MRINN row.

Run with the project venv: `.venv/bin/python -m ipykernel` or
`source .venv/bin/activate`.

# 1. Data

In [ ]:
import library_mrrnn as MR

MR.set_random_seed(42)

feats_prices = ["P_aFRR_pos", "P_mFRR_pos", "P_aFRR_neg", "P_mFRR_neg",
                "P_aFRR_pos_MOL", "P_aFRR_neg_MOL",
                "P_ID15_nemo", "P_ID60_nemo", "P_DA_nemo"]
feats_capacities = ["L_ID15", "L_ID60", "L_DA"]
feats_volume = ["system_imbalance", "E_aFRR_pos", "E_mFRR_pos",
                "E_aFRR_neg", "E_mFRR_neg"]
label = ["imbalance_price"]

path = str(MR.MRINN_ROOT / "Data" / "imbalance_data.csv")
regelzonen_data, feats = MR.load_data(feats_prices, feats_capacities,
                                      feats_volume, label, path)
print(MR.MRINN_ROOT, len(regelzonen_data))

# 2. Data Splitting & Scaling

`T` is the input length -- the number of 15-minute steps the recurrent encoder
sees. `T = 1` reproduces MRINN's setup and is the control for the sweep.

In [ ]:
train_range = ("2022-01-01 00:00:00+00:00", "2025-05-01 00:00:00+00:00")
val_range   = ("2025-05-01 00:00:00+00:00", "2025-09-01 00:00:00+00:00")
test_range  = ("2025-09-01 00:00:00+00:00", "2026-01-01 00:00:00+00:00")

T = 8
lags = list(range(1, T + 1))
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]

df_train, df_val, df_test = MR.split_data(regelzonen_data, train_range,
                                          val_range, test_range)
tr_s, va_s, te_s, feature_names = MR.shift_data(
    df_train, df_val, df_test, label[0], feats, [], lags, [])

# One scaler per signal across all its lags, so the time axis is not warped.
# Swap for MR.scale_data to reproduce MRINN's per-column scaling exactly.
X_train, X_val, X_test, y_train, y_val, y_test, y_scaler = \
    MR.scale_data_shared_lags(tr_s, va_s, te_s, feature_names, label)

print(f"T={T}  train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

# 3. Setting Parameters

C0-C10 are the regulatory constants (price floors, imbalance thresholds,
scarcity cap). They must be pushed through the same scaling as the features
or every gate saturates.

In [ ]:
Cs = MR.scaled_params(df_train, feats_prices, feats_capacities)
C_kwargs = {f"C{i}": Cs[i] for i in range(11)}
C_kwargs

# 4. Market-Rule-Informed LSTM (MR-LSTM)

`CASE_FOR_OUTPUT` selects the ablation: `ImbalancePrice` is the full model;
`ReservePriceIndex`, `ExchangePriceIndex` and `ScarcityFunction` attach the
quantile head to a single price component.

Keep `num_layer=1`: stacking two recurrent layers costs ~15k parameters, more
than the MLP baseline, which forfeits the efficiency argument.

In [ ]:
CASE_FOR_OUTPUT = "ImbalancePrice"

model_lstm, history_lstm = MR.build_MRRNN(
    CASE_FOR_OUTPUT, X_train, y_train, X_val, y_val,
    hidden_units=8, num_layer=1, epochs=50,
    **C_kwargs,
    cell="lstm", lags=lags, quantiles=quantiles,
    checkpoint_path=f"best_mrlstm_T{T}.keras",
)

In [ ]:
yqs_lstm = MR.make_inference(X_test, lags, quantiles,
                             checkpoint_path=f"best_mrlstm_T{T}.keras")
res_lstm = MR.evaluate_performance(y_test, yqs_lstm, quantiles, y_scaler)

# 5. Market-Rule-Informed GRU (MR-GRU)

Same model, cheaper cell (~4.9k vs ~5.8k parameters). Given that parameter
efficiency is the headline claim, this may well be the stronger result.

In [ ]:
model_gru, history_gru = MR.build_MRRNN(
    CASE_FOR_OUTPUT, X_train, y_train, X_val, y_val,
    hidden_units=8, num_layer=1, epochs=50,
    **C_kwargs,
    cell="gru", lags=lags, quantiles=quantiles,
    checkpoint_path=f"best_mrgru_T{T}.keras",
)

In [ ]:
yqs_gru = MR.make_inference(X_test, lags, quantiles,
                            checkpoint_path=f"best_mrgru_T{T}.keras")
res_gru = MR.evaluate_performance(y_test, yqs_gru, quantiles, y_scaler)

# 6. Comparison

MRINN's published row (Table 3): AQL 20.70, AQCR 0.00, MAE 49.36,
RMSE 277.33, 1.8k params. Expect MAE **parity**, not a win -- the literature
(Narajewski, O'Connor, Ganesh & Bunn) consistently finds recurrent models
matched by linear baselines on point accuracy while winning on density
quality. Judge on AQL, AQCE/AIW calibration and parameter count.

In [ ]:
import pandas as pd

# returns (metrics, yqs, y_true); te_s is unscaled -- naive_baseline scales internally
naive_res, _, _ = MR.naive_baseline(te_s, label[0], quantiles, y_scaler=y_scaler)

rows = {
    "Naive (persistence)": {**{k: naive_res[k] for k in ["AQL", "AQCR", "MAE", "RMSE"]},
                            "Params": 0},
    "MRINN (published)": {"AQL": 20.70, "AQCR": 0.00, "MAE": 49.36,
                          "RMSE": 277.33, "Params": 1817},
    "MR-LSTM": {**{k: res_lstm[k] for k in ["AQL", "AQCR", "MAE", "RMSE"]},
                "Params": model_lstm.count_params()},
    "MR-GRU":  {**{k: res_gru[k] for k in ["AQL", "AQCR", "MAE", "RMSE"]},
                "Params": model_gru.count_params()},
}
pd.DataFrame(rows).T[["AQL", "AQCR", "MAE", "RMSE", "Params"]]

In [ ]:
MR.plot_train_val_loss(history_lstm)

# 7. Input-length sweep

The point of the recurrent encoder. MRINN's own scaling analysis says the
optimal input length grows with the forecast horizon, so this is where a
temporal encoder should earn its parameters.

`T = 1` is the control: it should land near MRINN. If it doesn't, the
pipeline is wrong, not the model.

Note row counts differ slightly across `T` (`shift_data` drops `T` NaN rows
per split), so state `T` alongside any absolute loss.

In [ ]:
sweep = []

for T_i in [1, 4, 8, 16, 32]:
    lags_i = list(range(1, T_i + 1))
    tr_i, va_i, te_i, names_i = MR.shift_data(
        df_train, df_val, df_test, label[0], feats, [], lags_i, [])
    Xtr_i, Xva_i, Xte_i, ytr_i, yva_i, yte_i, ys_i = \
        MR.scale_data_shared_lags(tr_i, va_i, te_i, names_i, label)

    for cell in ["lstm", "gru"]:
        ckpt = f"sweep_{cell}_T{T_i}.keras"
        m, _ = MR.build_MRRNN(
            CASE_FOR_OUTPUT, Xtr_i, ytr_i, Xva_i, yva_i,
            hidden_units=8, num_layer=1, epochs=50,
            **C_kwargs, cell=cell, lags=lags_i, quantiles=quantiles,
            checkpoint_path=ckpt,
        )
        yq = MR.make_inference(Xte_i, lags_i, quantiles, checkpoint_path=ckpt)
        r = MR.evaluate_performance(yte_i, yq, quantiles, ys_i, verbose=False)
        sweep.append({"T": T_i, "cell": cell, "n_test": len(Xte_i),
                      "params": m.count_params(),
                      **{k: r[k] for k in ["AQL", "AQCR", "MAE", "RMSE", "AQCE", "AIW"]}})

pd.DataFrame(sweep)